<a href="https://colab.research.google.com/github/VikaSvyat/DI_Bootcamp/blob/main/Week13/BERT_Exercises_XP_Day3_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Week 13. LLM : Large Language Model - BERT


https://octopus.developers.institute/courses/collection/124/course/724/section/1978/chapter/4489

# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [1]:
# Optional setup: install dependencies if they are missing in your environment.
%pip install -q transformers torch


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "I'm trying to understand something"
print(sample_sentence)


I'm trying to understand something


In [6]:
tokens = tokenizer.tokenize(sample_sentence)
print(tokens)
print(len(tokens))

['i', "'", 'm', 'trying', 'to', 'understand', 'something']
7


In [7]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # TODO: adjust if your sentence needs more room - Don't need
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | i            |  1045
    2 | '            |  1005
    3 | m            |  1049
    4 | trying       |  2667
    5 | to           |  2000
    6 | understand   |  3305
    7 | something    |  2242
    8 | [SEP]        |   102
    9 | [PAD]        |     0
   10 | [PAD]        |     0
   11 | [PAD]        |     0
   12 | [PAD]        |     0
   13 | [PAD]        |     0
   14 | [PAD]        |     0
   15 | [PAD]        |     0
   16 | [PAD]        |     0
   17 | [PAD]        |     0
   18 | [PAD]        |     0
   19 | [PAD]        |     0
   20 | [PAD]        |     0
   21 | [PAD]        |     0
   22 | [PAD]        |     0
   23 | [PAD]        |     0

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens (index, token): [(0, '[CLS]'), (8, '[SEP]'), (9, '[PAD]'), (10, '[PAD]'), (11, '[PAD]'), (12, '[PAD]'), (13, '[PAD]'), (14, '[PAD]'), (15, '[PAD]')

### Exercise 1 reflection
-  Describe how [CLS] and [SEP] behave inside the encoder.

[CLS] — the “summary token”

Position is always at index 0

It collects information from all other tokens

After passing through all encoder layers, its final embedding becomes a summary of the whole sentence

So over layers:
 [CLS] becomes a compressed representation of the entire sequence

[SEP] — the “boundary marker”

Marks the end of a sentence. Also separates two sentences (for tasks like QA)

It tells the model: “This is where one segment ends.”


-  Explain how the attention mask hides padded positions from self-attention.

Real tokens → get probability

[PAD] → gets exactly 0

The attention mask works by:

Adding −∞ to padded positions

Softmax turns that into 0 probability

PAD tokens contribute nothing to representations

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [9]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "It was drawn out and a bit boring, but by the end, I enjoyed it more or less."
prediction = sentiment_pipeline(sentence)
prediction


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9991078972816467}]

### Exercise 2 reflection
-  Does the predicted label match your expectation? Why or why not?

My sentence:

First part → negative: “drawn out”, “boring”

Second part → positive: “I enjoyed it”, but mot much

Overall → mixed / slightly positive

So a POSITIVE label does make sense, but it’s not obvious or strong.
I think that model has clear labels (positive/negative) and mixed sentences are forced into one category.

- How confident is the model and what does the score tell you?

~99.91% confidence

This score actually not means “this is 99.9% objectively correct”,
it means: "the model is very certain given its training", it’s the softmax probability of the predicted class

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        # '''TODO: load the tokenizer/model and move the model to the proper device.'''
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.eval() #disables randomization
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.max_length = max_length
        # raise NotImplementedError("Initialize tokenizer, model, and device here.")

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        # '''TODO: clean the text, tokenize, and return tensors ready for inference.'''
        encoding = self.tokenizer(
                    text,
                    padding="max_length",
                    truncation=True,
                    max_length=self.max_length,
                    return_tensors="pt"
                )
        encoding = {k: v.to(self.device) for k, v in encoding.items()}
        return encoding
        # raise NotImplementedError("Return a dict of tensors produced by the tokenizer.")

    def predict(self, text: str) -> Dict[str, float]:
        # '''TODO: run a forward pass, apply softmax, and return a label plus probability.'''
        inputs = self.preprocess(text)
        with torch.no_grad(): #disables learning
          outputs = self.model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        pred_id = torch.argmax(probs, dim=-1).item()
        #find the best class
        score = probs[0, pred_id].item()
        label = self.model.config.id2label[pred_id]
        return {
            "label": label,
            "score": score
        }
        # raise NotImplementedError("Add inference and post-processing logic.")


In [6]:
# TODO: instantiate your analyzer and test several sentences once the class is ready.
analyzer = BERTSentimentAnalyzer()
samples = [
    "I love this movie!",
    "This was terrible...",
    "It was okay, not great"
]
for text in samples:
    print(text)
    print(analyzer.predict(text))


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

I love this movie!
{'label': 'POSITIVE', 'score': 0.9998775720596313}
This was terrible...
{'label': 'NEGATIVE', 'score': 0.9996702671051025}
It was okay, not great
{'label': 'NEGATIVE', 'score': 0.9948429465293884}


## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [7]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        '''TODO: load the tokenizer and model, and detect the available device.'''
        raise NotImplementedError("Initialize tokenizer, model, and device.")

    def recognize(self, text: str):
        '''TODO: tokenize the text, run the model, map predictions to BIO labels, and merge word pieces.'''
        raise NotImplementedError("Return structured entities.")

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        # '''TODO: load the tokenizer and model, and detect the available device.'''
        # raise NotImplementedError("Initialize tokenizer, model, and device.")

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()

    def recognize(self, text: str):
      #  '''TODO: tokenize the text, run the model, map predictions to BIO labels, and merge word pieces.'''
      #   raise NotImplementedError("Return structured entities.")

        # tokenization with offsets
        encoding = self.tokenizer(
            text,
            return_tensors="pt",
            return_offsets_mapping=True,
            truncation=True
        )
        offsets = encoding.pop("offset_mapping")[0]
        encoding = {k: v.to(self.device) for k, v in encoding.items()}
        #disables learning
        with torch.no_grad():
            outputs = self.model(**encoding)

        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)[0]

        tokens = self.tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])

        entities = []
        current_entity = None

        for i, (token, pred_id) in enumerate(zip(tokens, predictions)):
            label = self.model.config.id2label[pred_id.item()]
            start, end = offsets[i].tolist()

            # skip special tokens
            if start == end:
                continue

            # skip ##
            if token.startswith("##"):
                token = token[2:]

            if label.startswith("B-"):
                # start new entity
                if current_entity:
                    entities.append(current_entity)

                current_entity = {
                    "text": text[start:end],
                    "entity": label[2:],
                    "start": start,
                    "end": end
                }

            elif label.startswith("I-") and current_entity:
                # continue with entity
                current_entity["text"] += text[start:end]
                current_entity["end"] = end

            else:
                # closing entity
                if current_entity:
                    entities.append(current_entity)
                    current_entity = None

        # entity wasn't closed
        if current_entity:
            entities.append(current_entity)

        return entities

In [8]:
# TODO: instantiate the recognizer and test it on text that includes people, places, or organizations.
ner = BERTNamedEntityRecognizer()
sample_text = "The Company Developers Institute was started in Tel Aviv and creates courses for studying AI"
ner.recognize(sample_text)


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'text': 'CompanyDevelopersInstitute',
  'entity': 'ORG',
  'start': 4,
  'end': 32},
 {'text': 'TelAviv', 'entity': 'LOC', 'start': 48, 'end': 56},
 {'text': 'AI', 'entity': 'MISC', 'start': 90, 'end': 92}]

## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | Encoder-only Transformer (bidirectional attention) | Decoder-only Transformer (autoregressive, left-to-right) |
| Primary purpose | Understand language (encoding meaning of text) | Generate language (predict next token) |
| Typical use cases | Classification, sentiment analysis, NER, search, embeddings | Chatbots, text generation, code generation, completion |
| Strengths | Strong at understanding context both directions; great for labeling tasks | Strong at generating coherent long text; flexible general-purpose generation |
| Weaknesses | Cannot generate long text naturally | Less efficient for deep bidirectional understanding tasks like classification without fine-tuning |


## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:
1. Describe how BERT encodes queries and documents.

 BERT is used to convert both the user’s question (query) and all documents (knowledge base) into dense numerical vectors (embeddings).

BERT processes each text and produces an embedding that captures its semantic meaning, not just keywords. This means similar meanings produce similar vectors even if words differ.

2. Explain how those embeddings are stored and searched in a vector database.

Once generated, these embeddings are stored in a vector database.

When a user asks a question: the query is also converted into a vector and the system searches for nearest vectors using similarity metrics

This allows retrieval of documents that are meaning-wise closest, not just keyword matches.

3. Outline how the retrieved passages are handed to a generative model like GPT.

After retrieval, the system selects the top-k most relevant passages.

These passages are then passed into a generative model like GPT:

the model receives both the user question and the retrieved context from BERT embeddings

Then it generates a final answer grounded in that retrieved information, reducing hallucinations and improving factual accuracy.

4. Provide a concrete application example (industry or product) where RAG with BERT makes sense.

A common application is in customer support systems for large companies.

For example:

A user asks: “How do I reset my account password?”

BERT retrieves relevant help-center articles about password reset steps

A GPT-style model generates a clear, human-friendly answer using those articles